In [ ]:
# ICT-4442 Phase 2: TPU-v5e ResNet-50 + YOLOv8 benchmark
# TPU mode trains ResNet-50 with PyTorch/XLA. YOLOv8 is skipped because Ultralytics does not support TPU training.
import os
import sys
import json
import random
import shutil
import platform
import subprocess
from pathlib import Path

# TPU runtimes normally provide torch_xla. Install only if the import is unavailable.
try:
    import torch_xla.core.xla_model as xm
    import torch_xla.runtime as xr
    TPU_AVAILABLE = xr.xla_device_hw() == "TPU"
except ImportError:
    xm = None
    TPU_AVAILABLE = False

try:
    import ultralytics
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.3.0"])
    import ultralytics

import numpy as np
import pandas as pd
import yaml
import torch
from PIL import Image
from ultralytics import YOLO

SEED = 42
FAST_MODE = True
MAX_IMAGES = {"train": 5000, "val": 1000, "test": 1000} if FAST_MODE else {"train": None, "val": None, "test": None}
YOLO_EPOCHS = 5 if FAST_MODE else 50
RESNET_EPOCHS = 3 if FAST_MODE else 15
RESNET_BATCH_SIZE = 64 if TPU_AVAILABLE else (128 if torch.cuda.is_available() else 16)
CLASS_NAMES = ["D00_Longitudinal", "D10_Transverse", "D20_Alligator", "D40_Pothole"]
CLASS_CODES = {0: "D00", 1: "D10", 2: "D20", 3: "D40"}
DEVICE = xm.xla_device() if TPU_AVAILABLE else torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = Path("/kaggle/working/rdd2022_phase2" if Path("/kaggle/working").exists() else "/content/rdd2022_phase2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print({"python": platform.python_version(), "torch": torch.__version__, "ultralytics": ultralytics.__version__, "tpu_available": TPU_AVAILABLE, "device": str(DEVICE), "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, "max_images": MAX_IMAGES, "resnet_epochs": RESNET_EPOCHS, "resnet_batch": RESNET_BATCH_SIZE})

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
配置生成成功: rdd2022.yaml
Ultralytics 8.3.248 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
                                                       CUDA:1 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=rdd2022.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout

## Dataset contract

RDD2022 must contain four road-damage classes: `D00`, `D10`, `D20`, and `D40`. The dataset must include images and YOLO labels, with matching `train`, `val`, and `test` splits. Raw images without annotations cannot train or evaluate this project.

This notebook reports YOLO detection metrics and ResNet-50 crop-classification metrics separately. It does not invent severity, DSI, segmentation masks, or a combined score.

In [ ]:
# Locate a labelled dataset and validate its four-class YOLO annotations.
CANDIDATE_ROOTS = [
    Path("/kaggle/input/rdd-2022/RDD_SPLIT"),
    Path("/kaggle/input/rdd2022/RDD_SPLIT"),
    Path("/content/RDD_SPLIT"),
    Path("/content/rdd2022"),
    Path("RDD_SPLIT"),
]
DATA_ROOT = next((path for path in CANDIDATE_ROOTS if path.exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError("No labelled RDD2022 root found. Upload the dataset and add its path to CANDIDATE_ROOTS.")

SPLITS = {}
for split in ("train", "val", "test"):
    image_dir = DATA_ROOT / split / "images"
    label_dir = DATA_ROOT / split / "labels"
    if not image_dir.exists() or not label_dir.exists():
        raise FileNotFoundError(f"Missing {split}/images or {split}/labels under {DATA_ROOT}")
    SPLITS[split] = {"images": image_dir, "labels": label_dir}

def image_index(image_dir):
    result = {}
    for path in image_dir.rglob("*"):
        if path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
            result.setdefault(path.stem, path)
    return result

def read_yolo_label(path):
    annotations = []
    for line_number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        fields = line.split()
        if len(fields) != 5:
            raise ValueError(f"{path}:{line_number} must have 5 YOLO fields")
        class_id = int(fields[0])
        values = list(map(float, fields[1:]))
        if class_id not in CLASS_CODES or not all(0 < value < 1 for value in values):
            raise ValueError(f"Invalid annotation at {path}:{line_number}")
        annotations.append((class_id, *values))
    return annotations

manifest = []
for split, paths in SPLITS.items():
    images = image_index(paths["images"])
    labels = sorted(paths["labels"].rglob("*.txt"))
    if not labels:
        raise RuntimeError(f"No labels found in {paths['labels']}")
    for label_path in labels:
        image_path = images.get(label_path.stem)
        if image_path is None:
            raise FileNotFoundError(f"No matching image for {label_path}")
        annotations = read_yolo_label(label_path)
        for box_index, annotation in enumerate(annotations):
            manifest.append({"split": split, "image": str(image_path), "label": str(label_path), "box_index": box_index, "class_id": annotation[0], "class_name": CLASS_CODES[annotation[0]], "cx": annotation[1], "cy": annotation[2], "w": annotation[3], "h": annotation[4]})

manifest = pd.DataFrame(manifest)
manifest.to_csv(OUTPUT_DIR / "verified_annotation_manifest.csv", index=False)
print("Dataset:", DATA_ROOT)
print("Images:", manifest.image.nunique(), "Boxes:", len(manifest))
print(pd.crosstab(manifest.split, manifest.class_name))

In [ ]:
# Build a deterministic, class-covered subset for rapid Phase 2 training.
# The original dataset is never modified; both models use this generated subset.
subset_root = OUTPUT_DIR / "rdd2022_subset"
if subset_root.exists():
    shutil.rmtree(subset_root)
subset_splits = {}
selected_records = []

for split in ("train", "val", "test"):
    split_manifest = manifest[manifest.split == split].copy()
    image_classes = split_manifest.groupby("image")["class_id"].apply(lambda values: set(values.tolist())).to_dict()
    target = MAX_IMAGES[split] or len(image_classes)
    rng = random.Random(SEED + len(split))
    selected_images = set()
    # Guarantee at least one image for every class present in the source split.
    for class_id in range(4):
        candidates = [image for image, classes in image_classes.items() if class_id in classes]
        rng.shuffle(candidates)
        if candidates:
            selected_images.add(candidates[0])
    remaining = [image for image in image_classes if image not in selected_images]
    rng.shuffle(remaining)
    selected_images.update(remaining[:max(0, target - len(selected_images))])
    if len(selected_images) < min(target, len(image_classes)):
        raise RuntimeError(f"Could not create requested {split} subset")

    subset_image_dir = subset_root / split / "images"
    subset_label_dir = subset_root / split / "labels"
    subset_image_dir.mkdir(parents=True, exist_ok=True)
    subset_label_dir.mkdir(parents=True, exist_ok=True)
    image_rows = split_manifest.drop_duplicates("image")
    for index, row in enumerate(image_rows[image_rows.image.isin(selected_images)].itertuples(index=False), 1):
        source_image = Path(row.image)
        source_label = Path(row.label)
        unique_stem = f"{split}_{index:06d}_{source_image.stem}"
        shutil.copy2(source_image, subset_image_dir / f"{unique_stem}{source_image.suffix.lower()}")
        shutil.copy2(source_label, subset_label_dir / f"{unique_stem}.txt")
        selected_records.append({"split": split, "image": str(subset_image_dir / f"{unique_stem}{source_image.suffix.lower()}"), "label": str(subset_label_dir / f"{unique_stem}.txt")})

subset_map = pd.DataFrame(selected_records)
subset_manifest = manifest[manifest.apply(lambda row: row.image in set(subset_map[subset_map.split == row.split].image), axis=1)].copy()
# Reconstruct manifest coordinates from copied labels so crop classification uses the subset paths.
subset_rows = []
for record in selected_records:
    for box_index, annotation in enumerate(read_yolo_label(Path(record["label"]))):
        subset_rows.append({"split": record["split"], "image": record["image"], "label": record["label"], "box_index": box_index, "class_id": annotation[0], "class_name": CLASS_CODES[annotation[0]], "cx": annotation[1], "cy": annotation[2], "w": annotation[3], "h": annotation[4]})
manifest = pd.DataFrame(subset_rows)
SPLITS = {split: {"images": subset_root / split / "images", "labels": subset_root / split / "labels"} for split in ("train", "val", "test")}
manifest.to_csv(OUTPUT_DIR / "subset_annotation_manifest.csv", index=False)
print("Subset images:", manifest.groupby("split").image.nunique().to_dict())
print("Subset boxes:", len(manifest))
print(pd.crosstab(manifest.split, manifest.class_name))

In [ ]:
# YOLOv8n is CUDA/CPU-only in this notebook. Ultralytics does not support TPU-v5e training.
if TPU_AVAILABLE:
    yolo_metrics = {"model": "YOLOv8n", "task": "object_detection", "status": "skipped", "reason": "Ultralytics YOLOv8 training is not TPU/XLA compatible. Run this cell on a CUDA GPU for YOLO results."}
    (OUTPUT_DIR / "yolov8_test_metrics.json").write_text(json.dumps(yolo_metrics, indent=2), encoding="utf-8")
    print(yolo_metrics)
else:
    data_yaml = OUTPUT_DIR / "rdd2022_subset.yaml"
    data_yaml.write_text(yaml.safe_dump({
        "path": str(subset_root), "train": "train/images", "val": "val/images", "test": "test/images",
        "names": {0: CLASS_NAMES[0], 1: CLASS_NAMES[1], 2: CLASS_NAMES[2], 3: CLASS_NAMES[3]},
    }, sort_keys=False), encoding="utf-8")
    model = YOLO("yolov8n.pt")
    model.train(data=str(data_yaml), epochs=YOLO_EPOCHS, imgsz=512, batch=-1, device=0 if torch.cuda.is_available() else "cpu", workers=4 if torch.cuda.is_available() else 0, amp=torch.cuda.is_available(), cache="ram" if torch.cuda.is_available() else False, seed=SEED, patience=3 if FAST_MODE else 10, project=str(OUTPUT_DIR), name="yolov8n_rdd2022_subset", exist_ok=True, plots=True, save=True)
    yolo_test = model.val(data=str(data_yaml), split="test", imgsz=512, batch=-1, device=0 if torch.cuda.is_available() else "cpu", plots=True)
    box = yolo_test.box
    yolo_metrics = {"model": "YOLOv8n", "task": "object_detection", "status": "completed", "test": {"precision": float(box.mp), "recall": float(box.mr), "mAP50": float(box.map50), "mAP50_95": float(box.map)}}
    (OUTPUT_DIR / "yolov8_test_metrics.json").write_text(json.dumps(yolo_metrics, indent=2), encoding="utf-8")
    print(yolo_metrics)

In [ ]:
# Save YOLO test predictions when running on CUDA/CPU. TPU mode records the skip reason.
if TPU_AVAILABLE:
    print("YOLO prediction skipped: Ultralytics prediction is not TPU/XLA compatible.")
else:
    model.predict(source=str(SPLITS["test"]["images"]), imgsz=512, conf=0.25, device=0 if torch.cuda.is_available() else "cpu", save=True, project=str(OUTPUT_DIR), name="yolov8n_test_predictions", exist_ok=True, verbose=False)
    print("Prediction images:", OUTPUT_DIR / "yolov8n_test_predictions")

In [ ]:
# ResNet-50 classifier on verified damage crops; loaders are TPU-safe.
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]
train_transform = transforms.Compose([transforms.Resize((224, 224)), transforms.RandomHorizontalFlip(), transforms.ColorJitter(brightness=0.15, contrast=0.15), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class DamageCropDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True); self.transform = transform
    def __len__(self): return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row.image) as image:
            image = image.convert("RGB"); width, height = image.size
            box = (max(0, int((row.cx - row.w / 2) * width)), max(0, int((row.cy - row.h / 2) * height)), min(width, int((row.cx + row.w / 2) * width)), min(height, int((row.cy + row.h / 2) * height)))
            return self.transform(image.crop(box)), int(row.class_id)

crop_frames = {split: manifest[manifest.split == split] for split in ("train", "val", "test")}
crop_datasets = {split: DamageCropDataset(crop_frames[split], train_transform if split == "train" else eval_transform) for split in crop_frames}
counts = crop_frames["train"].class_id.value_counts().reindex(range(4), fill_value=0)
if (counts == 0).any(): raise RuntimeError(f"Missing training classes: {counts.to_dict()}")
sample_weights = crop_frames["train"].class_id.map({class_id: 1.0 / count for class_id, count in counts.items()}).to_numpy()
sampler = WeightedRandomSampler(torch.as_tensor(sample_weights, dtype=torch.double), len(sample_weights), replacement=True)
loader_args = {"batch_size": RESNET_BATCH_SIZE, "num_workers": 0 if TPU_AVAILABLE else (4 if torch.cuda.is_available() else 0), "pin_memory": torch.cuda.is_available(), "persistent_workers": False}
crop_loaders = {"train": DataLoader(crop_datasets["train"], sampler=sampler, **loader_args), "val": DataLoader(crop_datasets["val"], shuffle=False, **loader_args), "test": DataLoader(crop_datasets["test"], shuffle=False, **loader_args)}
print({split: len(dataset) for split, dataset in crop_datasets.items()}, "batch=", RESNET_BATCH_SIZE)

In [ ]:
# Train/evaluate ResNet-50 on TPU-v5e with PyTorch/XLA, or CUDA/CPU fallback.
resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
resnet.fc = torch.nn.Linear(resnet.fc.in_features, 4)
resnet = resnet.to(DEVICE)
resnet_device = DEVICE
optimizer = torch.optim.AdamW(resnet.parameters(), lr=2e-4, weight_decay=1e-4)
criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.05)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available() and not TPU_AVAILABLE)

def evaluate_classifier(loader):
    resnet.eval(); targets = []; predictions = []
    with torch.inference_mode():
        for images, labels in loader:
            logits = resnet(images.to(resnet_device))
            predictions.extend(logits.argmax(1).cpu().numpy()); targets.extend(labels.numpy())
            if TPU_AVAILABLE: xm.mark_step()
    precision, recall, f1, support = precision_recall_fscore_support(targets, predictions, labels=[0, 1, 2, 3], average=None, zero_division=0)
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(targets, predictions, labels=[0, 1, 2, 3], average="macro", zero_division=0)
    return {"accuracy": accuracy_score(targets, predictions), "macro_precision": macro_precision, "macro_recall": macro_recall, "macro_f1": macro_f1, "per_class": {CLASS_NAMES[index]: {"precision": float(precision[index]), "recall": float(recall[index]), "f1": float(f1[index]), "support": int(support[index])} for index in range(4)}, "confusion_matrix": confusion_matrix(targets, predictions, labels=[0, 1, 2, 3]).tolist(), "samples": len(targets)}

best_f1 = -1.0; classifier_history = []; classifier_checkpoint = OUTPUT_DIR / "resnet50_best.pt"
for epoch in range(1, RESNET_EPOCHS + 1):
    resnet.train(); loss_sum = 0.0
    for images, labels in crop_loaders["train"]:
        images, labels = images.to(resnet_device), labels.to(resnet_device)
        optimizer.zero_grad(set_to_none=True)
        if TPU_AVAILABLE:
            loss = criterion(resnet(images), labels); loss.backward(); xm.optimizer_step(optimizer); xm.mark_step()
        else:
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()): loss = criterion(resnet(images), labels)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        loss_sum += loss.item() * images.size(0)
    validation = evaluate_classifier(crop_loaders["val"])
    classifier_history.append({"epoch": epoch, "train_loss": loss_sum / len(crop_datasets["train"]), "validation": validation})
    print(classifier_history[-1])
    if validation["macro_f1"] > best_f1:
        best_f1 = validation["macro_f1"]
        torch.save({"model": resnet.state_dict(), "classes": CLASS_NAMES, "seed": SEED, "device": str(resnet_device)}, classifier_checkpoint)

resnet.load_state_dict(torch.load(classifier_checkpoint, map_location=resnet_device, weights_only=True)["model"])
resnet_metrics = {"model": "ResNet-50", "task": "damage_crop_classification", "device_type": "TPU/XLA" if TPU_AVAILABLE else str(resnet_device), "seed": SEED, "epochs": RESNET_EPOCHS, "classes": CLASS_NAMES, "history": classifier_history, "validation": evaluate_classifier(crop_loaders["val"]), "test": evaluate_classifier(crop_loaders["test"])}
(OUTPUT_DIR / "resnet50_test_metrics.json").write_text(json.dumps(resnet_metrics, indent=2), encoding="utf-8")
print(json.dumps(resnet_metrics["test"], indent=2))

## Phase 2 submission evidence

This notebook now provides the implementation evidence for two proposed models:

- **ResNet-50:** accuracy, macro precision, macro recall, macro F1, confusion matrix, checkpoint, and training history.
- **YOLOv8n:** precision, recall, mAP@0.5, mAP@0.5:0.95, checkpoint, plots, and test prediction images.

Still complete these report items from verified sources and actual team work:

1. Literature table with 8–10 papers.
2. Official dataset URL, version/date, license, and preprocessing description.
3. Signed contribution matrix and real commit IDs for all four members.
4. Risk table and final timeline for U-Net and transformer work.
5. Plagiarism and AI-content reports if required by faculty.

Do not report metrics unless the corresponding cells completed successfully and the JSON artifacts exist in the output directory.

In [ ]:
# Save environment metadata and package the verified Phase 2 evidence.
run_metadata = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "ultralytics": ultralytics.__version__,
    "cuda": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "seed": SEED,
    "dataset_root": str(DATA_ROOT),
    "class_names": CLASS_NAMES,
    "outputs": sorted(str(path.relative_to(OUTPUT_DIR)) for path in OUTPUT_DIR.rglob("*") if path.is_file()),
}
(OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2), encoding="utf-8")
archive_path = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("Evidence archive:", archive_path)
print("Do not upload raw images, credentials, or private dataset files to GitHub.")